In [ ]:
# !pip install -q h5py mat73 openpyxl
# !pip install -q numpy pandas matplotlib scipy torch

# # Note: avalanche-lib pulls in a couple of packages (e.g. proxsuite) that
# # need a C/C++ toolchain to build from source.
# #   - Linux:   sudo apt-get install -y build-essential   (then pip install avalanche-lib)
# #   - macOS:   xcode-select --install                    (then pip install avalanche-lib)
# #   - Windows: install "Visual C++ Build Tools" from
# #              https://visualstudio.microsoft.com/visual-cpp-build-tools/
# # (Earlier revisions of this cell only mentioned the Windows case; pick
# # whichever matches the machine you're actually running on.)

# import warnings
# warnings.filterwarnings("ignore")


In [ ]:
# !pip install -q avalanche-lib
# # If this fails to build, install a C/C++ toolchain first (see the note in
# # the previous cell for your OS), then re-run this line. The model
# # architecture and data loading below work fine even without Avalanche
# # installed (see the AVALANCHE_AVAILABLE guard later) -- you'll just be
# # limited to dataset preparation until it's installed.
# #   Linux/macOS: install build tools, then `pip install avalanche-lib`
# #   Windows:     install Visual C++ Build Tools, then `pip install avalanche-lib`


In [ ]:
import os, math, random, glob
from collections import defaultdict
from functools import lru_cache

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from scipy.io import loadmat, savemat
from scipy.signal import savgol_filter

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)


device: cpu


In [ ]:
# LOCAL DATASET PATH - Point to your EHUNAM folder.
# Override with an environment variable so this notebook is portable across
# machines/collaborators without editing the hardcoded default below, e.g.:
#   export EHUNAM_DATA_ROOT=/path/to/EHUNAM
DATA_ROOT = os.environ.get("EHUNAM_DATA_ROOT", r"/home/ab5-412/Downloads/Secret/28541225/EHUNAM")

# Verify the folder exists
if not os.path.exists(DATA_ROOT):
    raise ValueError(f"EHUNAM dataset folder not found at {DATA_ROOT}")

print(f"Using local EHUNAM dataset from: {DATA_ROOT}")
print(f"Found {len(glob.glob(os.path.join(DATA_ROOT, '*.mat')))} .mat files")

# Quick heads-up about Summary.xlsx/.mat -- Section 3 will look for these
# automatically and use them for fast, robust metadata if present (see the
# code review's recommended fix for the Environment-field parsing issue).
_summary_hint = [f for f in os.listdir(DATA_ROOT) if f.lower().startswith("summary")]
if not _summary_hint:
    _parent_hint = [f for f in os.listdir(os.path.dirname(DATA_ROOT.rstrip("/\\")))
                     if f.lower().startswith("summary")] if os.path.dirname(DATA_ROOT.rstrip("/\\")) else []
    _summary_hint = _parent_hint
print(f"Summary.xlsx/.mat found near DATA_ROOT: {_summary_hint or 'none -- will fall back to per-file .mat parsing (see Section 3)'}")


ValueError: EHUNAM dataset folder not found at /home/ab5-412/Downloads/Secret/28541225/EHUNAM

In [ ]:
# Skip download since we're using local data
print("Using local EHUNAM data - no download needed.")

In [ ]:
# Skip extraction since we're using local .mat files directly
print(f"\nDATA_ROOT = {DATA_ROOT}")
print("Ready to index .mat files.")

In [ ]:
# Set to True only for a quick offline pipeline check with synthetic data.
MOCK_MODE = False

# Real MC1 environments (per the EHUNAM paper) and which activities were
# actually recorded in each -- basement/storage/laboratory were only ever
# W/S/J; classroom/office also got the sit/sit-to-stand/fall activities
# this notebook's HEALTHCARE_ACTIVITIES filter cares about (see the
# per-domain class-coverage note in Section 3). The mock generator below
# mirrors this restriction on purpose, and also mirrors the real dataset's
# suspected metadata quirk -- Environment is deliberately left OUT of each
# individual .mat file's metadata and only recoverable from a Summary.xlsx
# file (see Section 3's Environment-resolution logic) -- so this doubles as
# an end-to-end test of the code-review fix, not just a schema smoke test.
ENVIRONMENT_SETS = {
    "Classroom":    ["MC1_01A", "MC1_01B"],
    "Office 1":     ["MC1_02"],
    "Basement Room": ["MC1_03"],
    "Storage Room": ["MC1_04"],
    "Laboratory":   ["MC1_05"],
}
ENV_ACTIVITIES = {
    "Classroom": ["W", "S", "J", "T", "G", "F"],
    "Office 1": ["W", "S", "J", "T", "G", "F"],
    "Basement Room": ["W", "S", "J"],
    "Storage Room": ["W", "S", "J"],
    "Laboratory": ["W", "S", "J"],
}
# rough real-world imbalance: walk/stand recorded far more often than
# sit-to-stand/fall/sit -- see the class-weighting note in Section 5.
ACTIVITY_FILE_COUNTS = {"W": 10, "S": 10, "J": 6, "T": 3, "G": 3, "F": 3}
N_SUBCARRIERS_RAW = 234  # 80 MHz occupied subcarriers per the EHUNAM paper


def _make_mock_measurement(path, activity, application, set_name, rx, people,
                            machine, status, number, n_frames, rng, include_environment=None):
    amp = rng.normal(loc=10, scale=2, size=(n_frames, N_SUBCARRIERS_RAW))
    phase = rng.uniform(-np.pi, np.pi, size=(n_frames, N_SUBCARRIERS_RAW))
    csi = (amp * np.exp(1j * phase)).astype(np.complex128)
    payload = {
        "CSI": csi, "Activity": activity, "Application": application,
        "Band": "5", "BW": "80", "Channel": "36",
        "N_Rx": 2, "N_People": 1, "People": people,
        "Occupied_SC": N_SUBCARRIERS_RAW, "Rx": rx, "Set": set_name,
        "Standard": "802.11ac", "Number": number,
    }
    if include_environment:  # deliberately usually left out -- see cell docstring
        payload["Environment"] = include_environment
    savemat(path, payload)


def build_mock_ehunam(root, seed=0, write_summary=True):
    """Writes a small set of schema-correct fake .mat files (+ a
    Summary.xlsx) so the rest of the notebook can run without the real
    (72GB) release, and so the Environment-resolution fix can be exercised
    end-to-end (individual files don't carry Environment; Summary.xlsx
    does)."""
    os.makedirs(root, exist_ok=True)
    rng = np.random.default_rng(seed)
    idx = 0
    summary_rows = []
    for env, sets in ENVIRONMENT_SETS.items():
        for set_name in sets:
            for act in ENV_ACTIVITIES[env]:
                for n in range(ACTIVITY_FILE_COUNTS[act]):
                    idx += 1
                    people = chr(ord('a') + (idx % 6))
                    fname = f"{set_name}_1_HAR_{people}_{act}_1_O_{idx:02d}.mat"
                    n_frames = int(rng.integers(300, 620))
                    _make_mock_measurement(
                        os.path.join(root, fname), activity=act, application="HAR",
                        set_name=set_name, rx="1", people=people, machine="1",
                        status="O", number=idx, n_frames=n_frames, rng=rng,
                        include_environment=None,  # <- the point of this mock
                    )
                    summary_rows.append({"Set": set_name, "Environment": env,
                                          "Application": "HAR", "Activity": act,
                                          "Filename": fname})
            # a couple of non-HAR / empty-room files per set, to exercise the
            # Application=="HAR" pre-filter in build_index
            for app, act in [("E", "E"), ("PC", "unknown")]:
                idx += 1
                fname = f"{set_name}_1_{app}_#_{act}_#_#_{idx:02d}.mat"
                _make_mock_measurement(
                    os.path.join(root, fname), activity=act, application=app,
                    set_name=set_name, rx="1", people="#", machine="#",
                    status="#", number=idx, n_frames=int(rng.integers(300, 500)), rng=rng,
                )
                summary_rows.append({"Set": set_name, "Environment": env,
                                      "Application": app, "Activity": act, "Filename": fname})

    if write_summary:
        pd.DataFrame(summary_rows).to_excel(os.path.join(root, "Summary.xlsx"), index=False)

    print(f"[mock] wrote {idx} synthetic .mat files "
          f"({'plus Summary.xlsx' if write_summary else 'no Summary.xlsx'}) to {root}")


if MOCK_MODE:
    DATA_ROOT = "/content/ehunam_mock"
    build_mock_ehunam(DATA_ROOT)

print(f"DATA_ROOT = {DATA_ROOT}")


In [ ]:
"""
EHUNAM loading / preprocessing pipeline.

Reads the .mat files as documented in the EHUNAM Scientific Data paper
(one measurement per file; complex CSI matrix + metadata fields such as
Activity, Environment, Occupied_SC, ...), builds a metadata index, picks
the largest-sample-count environment as the base ("Phase 1") task, and
turns each measurement into fixed-length amplitude/phase windows.

--- Code-review fixes applied in this cell ---
The previous version had a data-integrity bug: whenever a file's
`Environment` metadata failed to parse, it silently invented one via
`hash(filename) % 1000`. On the real release this fired for essentially
every file (the paper's own file-naming spec confirms Environment is never
encoded in the filename, so there's no legitimate way to "recover" it that
way), which turned a 5-environment problem into ~350 meaningless
single-file "environments" -- see the code review for the full trace and
its effect on training time/accuracy. This version instead:
  1. Never fabricates an Environment value.
  2. Prefers `Summary.xlsx` / `Summary.mat` (documented as shipping with
     the official release) for metadata when available -- fast, and it
     sidesteps the raw-`.mat` string-field parsing issue entirely.
  3. Otherwise resolves Environment per *Set* (Environment is constant
     within a Set per the paper's Table 4/5/6 structure) by propagating
     whatever resolves from any source -- a file's own metadata, the
     summary table, or a manual `SET_TO_ENVIRONMENT` override you can fill
     in yourself -- to every file sharing that Set.
  4. Filters to `Application == "HAR"` from the filename alone, *before*
     opening a file's (potentially large) CSI array, since only HAR
     measurements can contain the S/T/G/F/W activity codes this notebook
     trains on (avoids opening ~80% of the release for nothing).
  5. Any file whose Environment truly can't be resolved from any source is
     dropped, with a printed count -- not silently mislabeled.
"""
import os
import glob
import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy.signal import savgol_filter

try:
    import h5py
    _HAVE_H5PY = True
except ImportError:
    _HAVE_H5PY = False


# Fill this in yourself if you know the Set -> Environment mapping for your
# release (e.g. read off Summary.xlsx by hand, or the paper's Table 4/5/6)
# and want it to take priority over -- or fill gaps left by -- automatic
# resolution below. Example:
#   SET_TO_ENVIRONMENT = {"MC1_01A": "Classroom", "MC1_02": "Basement Room"}
SET_TO_ENVIRONMENT: dict = {}


def _looks_like_real_string(s):
    """Guards against a subtle failure mode: if a .mat field is some object
    scipy can't cleanly decode (e.g. a MATLAB `string`-type opaque object),
    `str(...)` on it can produce a non-empty but garbage repr (e.g.
    "<scipy.io.matlab...MatlabOpaque object at 0x7f...>"), which would
    previously have been accepted as a "real" value because it isn't an
    empty string. Reject anything that looks like a Python repr instead of
    real data."""
    return bool(s) and not s.startswith("<") and "object at 0x" not in s and s.lower() != "nan"


def _scalar(v):
    """.mat fields loaded via scipy come back wrapped in nested arrays;
    unwrap to a plain python scalar/string. Handles multi-element arrays gracefully."""
    arr = np.asarray(v)

    # Unwrap single-element wrappers
    while arr.ndim > 0 and arr.shape[0] == 1 and arr.ndim == 1:
        arr = arr[0]

    # If it's still an array with multiple elements, just take the first
    if isinstance(arr, np.ndarray) and arr.size > 1:
        arr = arr.flat[0]

    # Convert strings
    if isinstance(arr, np.ndarray) and arr.dtype.kind in "US":
        return str(arr.item() if arr.ndim == 0 else arr.flat[0])
    if isinstance(arr, (bytes, np.bytes_)):
        return arr.decode() if isinstance(arr, bytes) else str(arr)

    # Try to convert to scalar
    if isinstance(arr, np.ndarray):
        if arr.ndim == 0:
            return arr.item()
        elif arr.size == 1:
            return arr.flat[0]
        else:
            return arr.flat[0]  # Fall back to first element

    return arr


def load_mat_any(path):
    """MATLAB -v7 files load with scipy.io; -v7.3 (HDF5-based, common for
    large CSI matrices) need h5py. Try scipy first, fall back to h5py."""
    try:
        return loadmat(path), "scipy"
    except NotImplementedError:
        if not _HAVE_H5PY:
            raise RuntimeError(
                f"{path} looks like MATLAB v7.3 (HDF5) format; install h5py to read it."
            )
        f = h5py.File(path, "r")
        return f, "h5py"


def parse_filename_fields(path):
    """Best-effort parse of the 9 underscore-separated fields documented in
    the EHUNAM paper's "Data File Name" section, e.g.
    MC1_01A_1_HAR_e_J_#_#_01.mat -> campaign, set_num, rx, application,
    people, activity, machine, status, number. Never opens the file --
    used to cheaply skip files we don't need (non-HAR measurements) before
    paying the cost of loading a full CSI matrix. Fields whose value is the
    "#" placeholder (meaning "not applicable") are omitted."""
    stem = os.path.splitext(os.path.basename(path))[0]
    parts = stem.split("_")
    field_names = ["campaign", "set_num", "rx", "application", "people",
                   "activity", "machine", "status", "number"]
    return {k: v for k, v in zip(field_names, parts) if v and v != "#"}


def filename_set(path):
    """Field 1 + field 2 of the filename, e.g. 'MC1_01A' -- this is what the
    .mat file's own `Set` variable should equal, and it's the join key used
    to resolve Environment (see module docstring)."""
    fields = parse_filename_fields(path)
    campaign, set_num = fields.get("campaign"), fields.get("set_num")
    if campaign and set_num:
        return f"{campaign}_{set_num}"
    return None


def extract_activity_from_filename(filename):
    """Extract activity code from EHUNAM filename.
    Filenames typically contain activity codes: W, S, J, T, G, F, E
    e.g., MC1_01A_1_HAR_a_J_#_#_01.mat -> 'J'
    """
    basename = os.path.basename(filename)
    parts = basename.split('_')
    for part in parts:
        if len(part) == 1 and part in ['W', 'S', 'J', 'T', 'G', 'F', 'E']:
            return part
    return 'unknown'


# ---------------------------------------------------------------------------
# Summary.xlsx / Summary.mat -- fast, robust metadata source (preferred)
# ---------------------------------------------------------------------------
_SUMMARY_FILENAMES = ("summary.xlsx", "summary.mat")
_COLUMN_ALIASES = {
    "environment": ["Environment", "environment", "Location", "Scenario"],
    "set": ["Set", "set"],
}


def find_summary_file(root_dir, max_parent_levels=2):
    """Look for Summary.xlsx/.mat in root_dir and a couple of parent
    directories (the release sometimes ships it next to, rather than
    inside, the per-file .mat folder)."""
    candidates = []
    d = os.path.abspath(root_dir)
    for _ in range(max_parent_levels + 1):
        if os.path.isdir(d):
            for fname in os.listdir(d):
                if fname.lower() in _SUMMARY_FILENAMES:
                    candidates.append(os.path.join(d, fname))
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    candidates.sort(key=lambda p: 0 if p.lower().endswith(".xlsx") else 1)  # prefer .xlsx
    return candidates[0] if candidates else None


def _resolve_col(df, key):
    for cand in _COLUMN_ALIASES[key]:
        if cand in df.columns:
            return cand
    return None


def load_summary_table(root_dir):
    """Best-effort loader for the EHUNAM Summary.xlsx / Summary.mat file --
    the recommended (fast, robust) metadata source per the code review.
    Returns a DataFrame, or None if not found / not parseable, in which
    case the per-file .mat fallback below is used instead."""
    path = find_summary_file(root_dir)
    if path is None:
        print("[summary] no Summary.xlsx/.mat found near DATA_ROOT -- "
              "falling back to per-file .mat metadata parsing for Environment.")
        return None
    try:
        if path.lower().endswith(".xlsx"):
            df = pd.read_excel(path)
        else:
            # Summary.mat's table is very likely a MATLAB `table` object,
            # which (like MATLAB `string` arrays) scipy cannot deserialize.
            # Only plain array-shaped variables can be recovered this way.
            raw = loadmat(path)
            data = {k: np.asarray(v).squeeze().tolist() for k, v in raw.items()
                    if not k.startswith("__")}
            df = pd.DataFrame(data)
        df.columns = [str(c).strip() for c in df.columns]
        print(f"[summary] loaded {len(df)} rows from {path}")
        return df
    except Exception as e:
        print(f"[summary] found {path} but couldn't parse it "
              f"({type(e).__name__}: {e}); falling back to per-file .mat parsing.")
        return None


def summary_set_to_environment(summary_df):
    """Build a {Set: Environment} dict from the summary table, if it has
    recognizable columns for both."""
    if summary_df is None:
        return {}
    set_col = _resolve_col(summary_df, "set")
    env_col = _resolve_col(summary_df, "environment")
    if not set_col or not env_col:
        return {}
    out = {}
    for _, r in summary_df[[set_col, env_col]].dropna().iterrows():
        s, e = str(r[set_col]).strip(), str(r[env_col]).strip()
        if _looks_like_real_string(s) and _looks_like_real_string(e):
            out.setdefault(s, e)
    return out


# ---------------------------------------------------------------------------
# per-file reading
# ---------------------------------------------------------------------------
def read_measurement(path):
    """Returns a dict with keys: csi (complex ndarray [n_frames, n_subcarriers]),
    activity, environment, occupied_sc, rx, set, plus the raw metadata dict.

    `environment` may come back None here if it can't be resolved from the
    file's own metadata -- `build_index` fills it in afterwards from
    Summary.xlsx / Set-propagation / SET_TO_ENVIRONMENT, and drops the file
    if it truly can't be resolved. This function itself never fabricates a
    value (see module docstring)."""
    try:
        data, backend = load_mat_any(path)
    except Exception as e:
        print(f"Error loading {path}: {e}")
        raise

    try:
        if backend == "scipy":
            csi = np.asarray(data["CSI"])
            meta = {}
            for k, v in data.items():
                if not k.startswith("__") and k != "CSI":
                    try:
                        meta[k] = _scalar(v)
                    except Exception:
                        pass  # field genuinely couldn't be parsed -- resolved via fallbacks below
        else:
            csi_raw = data["CSI"][()]
            if isinstance(csi_raw, np.ndarray) and csi_raw.dtype.names:
                csi = csi_raw["real"] + 1j * csi_raw["imag"]
            else:
                csi = csi_raw
            csi = csi.T if csi.ndim == 2 else csi
            meta = {}
            for k in data.keys():
                if k == "CSI":
                    continue
                try:
                    meta[k] = _scalar(data[k][()])
                except Exception:
                    pass
            data.close()

        if csi.ndim != 2:
            raise ValueError(f"{path}: expected 2D CSI matrix, got shape {csi.shape}")

        occ = meta.get("Occupied_SC", csi.shape[1])
        occ = int(occ) if np.isscalar(occ) or isinstance(occ, (int, float, np.integer, np.floating)) else csi.shape[1]
        if 0 < occ < csi.shape[1]:
            # keep the centred occupied-subcarrier block, dropping guard/null bins
            start = (csi.shape[1] - occ) // 2
            csi = csi[:, start:start + occ]

        activity = str(meta.get("Activity", "")).strip()
        if not _looks_like_real_string(activity) or activity.lower() == 'unknown':
            activity = extract_activity_from_filename(path)

        environment = str(meta.get("Environment", "")).strip()
        if not _looks_like_real_string(environment) or environment.lower() == 'unknown':
            environment = None  # resolved later by build_index -- never fabricated here

        set_name = str(meta.get("Set", "")).strip()
        if not _looks_like_real_string(set_name) or set_name.lower() == 'unknown':
            set_name = filename_set(path) or "unknown"

        return {
            "csi": csi,
            "activity": activity,
            "environment": environment,
            "set": set_name,
            "n_frames": csi.shape[0],
            "n_subcarriers": csi.shape[1],
            "path": path,
            "meta": meta,
        }
    except Exception as e:
        print(f"Error processing {path}: {e}")
        raise


def build_index(root_dir, pattern="*.mat", require_application="HAR"):
    """Scans root_dir for .mat files and returns a per-file metadata
    DataFrame.

    Two changes from the original version (see code review):
      - Filters to `Application == require_application` straight from the
        filename before opening a file's CSI array at all (only HAR files
        can contain the S/T/G/F/W activities this notebook trains on, and
        for the real release this is ~5x fewer files to open).
      - Resolves Environment via Summary.xlsx (if found) and/or by
        propagating any value that resolves for a Set to every file sharing
        that Set, instead of ever fabricating one. Files whose Set never
        resolves an Environment from any source are dropped, and the count
        is printed so this can't fail silently.
    """
    summary_df = load_summary_table(root_dir)
    summary_env_map = summary_set_to_environment(summary_df)

    all_paths = sorted(glob.glob(os.path.join(root_dir, "**", pattern), recursive=True))
    n_total = len(all_paths)

    rows = []
    n_skipped_application = 0
    for path in all_paths:
        if require_application:
            fname_app = parse_filename_fields(path).get("application", "").upper()
            # Only skip on the filename check when we can actually read an
            # application code from it; if a filename doesn't match the
            # documented 9-field scheme, fall through to opening the file
            # rather than silently dropping something we can't judge.
            if fname_app and fname_app != require_application.upper():
                n_skipped_application += 1
                continue

        try:
            rec = read_measurement(path)
        except Exception as e:
            print(f"  [skip] {path}: {e}")
            continue
        rows.append({
            "path": rec["path"], "activity": rec["activity"],
            "environment": rec["environment"], "set": rec["set"],
            "n_frames": rec["n_frames"], "n_subcarriers": rec["n_subcarriers"],
        })

    index_df = pd.DataFrame(rows)
    print(f"Scanned {n_total} .mat files; skipped {n_skipped_application} "
          f"non-'{require_application}' files by filename alone (no CSI opened "
          f"for those); opened {len(index_df)} files.")

    if index_df.empty:
        return index_df

    # ---- Environment resolution: propagate per-Set, never fabricate ----
    set_to_env = dict(SET_TO_ENVIRONMENT)  # manual overrides take top priority
    for set_name, env in zip(index_df["set"], index_df["environment"]):
        if env and set_name not in set_to_env:
            set_to_env[set_name] = env
    for set_name, env in summary_env_map.items():
        set_to_env.setdefault(set_name, env)

    resolved_env = index_df["set"].map(set_to_env)
    n_unresolved = int(resolved_env.isna().sum())
    if n_unresolved:
        unresolved_sets = sorted(index_df.loc[resolved_env.isna(), "set"].unique())
        print(f"[environment] {n_unresolved} file(s) across {len(unresolved_sets)} "
              f"Set(s) have no resolvable Environment from any source (file "
              f"metadata / Summary.xlsx / SET_TO_ENVIRONMENT) and are being "
              f"dropped rather than mislabeled: {unresolved_sets[:10]}"
              f"{' ...' if len(unresolved_sets) > 10 else ''}\n"
              f"  -> If you know these Sets' environments (e.g. from the "
              f"paper's Table 4), add them to SET_TO_ENVIRONMENT at the top "
              f"of this cell and re-run.")
    index_df["environment"] = resolved_env
    index_df = index_df.dropna(subset=["environment"]).reset_index(drop=True)
    return index_df


def pick_base_domain(index_df, domain_col="environment"):
    """The 'feature with the most data points': the domain (here,
    Environment) with the largest total number of CSI frames. Returns
    (base_domain, ordered_domain_list, sizes) where ordered_domain_list
    starts with the base domain followed by the rest, largest-to-smallest,
    giving the task order fed into the continual-learning stream."""
    sizes = index_df.groupby(domain_col)["n_frames"].sum().sort_values(ascending=False)
    ordered = list(sizes.index)
    return ordered[0], ordered, sizes


# ------------------------------- preprocessing ------------------------------
def sanitize_phase(phase, window_length=7, polyorder=2):
    """Unwrap + linear-trend removal + light smoothing, roughly following the
    established CSI phase-sanitization recipe (unwrap -> remove the linear
    component introduced by sampling-time/frequency offsets -> denoise)."""
    unwrapped = np.unwrap(phase, axis=0)
    idx = np.arange(unwrapped.shape[0])
    # remove a per-subcarrier linear trend (least-squares fit) instead of raw phase
    A = np.stack([idx, np.ones_like(idx)], axis=1).astype(float)
    coefs, *_ = np.linalg.lstsq(A, unwrapped, rcond=None)
    trend = A @ coefs
    detrended = unwrapped - trend
    wl = min(window_length, detrended.shape[0] - (1 - detrended.shape[0] % 2))
    if wl >= 5 and wl % 2 == 1 and wl <= detrended.shape[0]:
        detrended = savgol_filter(detrended, wl, polyorder, axis=0)
    return detrended


def denoise_amplitude(amp, window_length=7, polyorder=2):
    wl = min(window_length, amp.shape[0] - (1 - amp.shape[0] % 2))
    if wl >= 5 and wl % 2 == 1 and wl <= amp.shape[0]:
        return savgol_filter(amp, wl, polyorder, axis=0)
    return amp


def preprocess_measurement(rec, target_subcarriers=None):
    csi = rec["csi"]
    amp = np.abs(csi)
    phase = np.angle(csi)
    amp = denoise_amplitude(amp)
    phase = sanitize_phase(phase)
    amp = (amp - amp.mean(0, keepdims=True)) / (amp.std(0, keepdims=True) + 1e-6)
    phase = (phase - phase.mean(0, keepdims=True)) / (phase.std(0, keepdims=True) + 1e-6)
    if target_subcarriers is not None and amp.shape[1] != target_subcarriers:
        # simple resample along the subcarrier axis to a common width so
        # windows from different bandwidth configs (20/40/80 MHz) can share
        # one model input size. NOTE (code review): this mixes genuinely
        # different RF configurations (20/80MHz, ACT/NCE chipsets) via naive
        # index interpolation, which doesn't preserve true frequency
        # mapping and is confounded with campaign/domain -- kept as-is here
        # since it's a modeling/methodology tradeoff, not a bug, but worth
        # a sanity check (e.g. train/test within one BW config) before
        # reporting cross-domain numbers as evidence of activity-general
        # learning rather than acquisition-setup learning.
        idx = np.linspace(0, amp.shape[1] - 1, target_subcarriers)
        amp = np.stack([np.interp(idx, np.arange(amp.shape[1]), amp[t]) for t in range(amp.shape[0])])
        phase = np.stack([np.interp(idx, np.arange(phase.shape[1]), phase[t]) for t in range(phase.shape[0])])
    return amp.astype(np.float32), phase.astype(np.float32)


def window_starts(n_frames, win_len=64, stride=32):
    """Sliding-window start-offset arithmetic, shared by `make_windows`
    (operates on already-loaded arrays, e.g. for the Section 4 plot) and
    `EHUNAMWindowDataset` (only needs the offsets, without loading full
    arrays) -- consolidated so the two copies can't drift out of sync
    (flagged as a maintainability item in the code review)."""
    return list(range(0, max(1, n_frames - win_len + 1), stride))


def make_windows(amp, phase, win_len=64, stride=32):
    windows = []
    for start in window_starts(amp.shape[0], win_len, stride):
        a = amp[start:start + win_len]
        p = phase[start:start + win_len]
        if a.shape[0] < win_len:
            pad = win_len - a.shape[0]
            a = np.pad(a, ((0, pad), (0, 0)), mode="edge")
            p = np.pad(p, ((0, pad), (0, 0)), mode="edge")
        windows.append(np.stack([a, p], axis=-1))  # (win_len, S, 2)
    return windows


In [ ]:
index_df = build_index(DATA_ROOT)
print(f"Indexed {len(index_df)} measurement files.")
index_df.head()


In [ ]:
# Keep the healthcare-relevant activities the paper focuses on: standing,
# sitting, walking, sit-to-stand transitions, falling. (Add "J"/"E" back in
# if you want jumping / empty-room negatives too.)
HEALTHCARE_ACTIVITIES = ["S", "T", "G", "F", "W"]
index_df = index_df[index_df["activity"].isin(HEALTHCARE_ACTIVITIES)].reset_index(drop=True)

# --- Sanity check (code review): Environment should now be a small,
# real set of physical locations, not hundreds of hash-derived fakes. ---
n_envs = index_df["environment"].nunique()
print(f"Resolved {n_envs} distinct environment(s):")
print(index_df["environment"].value_counts())
if n_envs > 15:
    print("\n[warning] That's a lot of 'environments' for a physical dataset "
          "with a handful of rooms -- if you're seeing this, Environment "
          "resolution likely isn't working as intended (see Section 3 / the "
          "code review). Check the SET_TO_ENVIRONMENT overrides and whether "
          "Summary.xlsx was found and parsed correctly above.")

base_domain, domain_order, domain_sizes = pick_base_domain(index_df, domain_col="environment")
print("\nBase (Phase-1) domain:", base_domain)
print("Full task order (largest \u2192 smallest):", domain_order)

fig, ax = plt.subplots(figsize=(6, 3.5))
domain_sizes.plot(kind="bar", ax=ax, color=["#4C72B0" if d == base_domain else "#8C8C8C" for d in domain_sizes.index])
ax.set_ylabel("total CSI frames")
ax.set_title("Frames per environment -- base task = largest bar")
plt.tight_layout(); plt.show()

# --- Per-domain class coverage (code review, section 3) ---
# EHUNAM's own protocol means some MC1 environments were only ever recorded
# doing a subset of these activities (e.g. basement/storage/lab never
# recorded sit/sit-to-stand/fall) -- this is a real property of the
# dataset, not a bug, but it means some domains' accuracy on the missing
# classes is undefined/degenerate rather than a meaningful transfer-failure
# signal. Surfacing it here so it's not silently misread later.
coverage = pd.crosstab(index_df["environment"], index_df["activity"])
print("\nActivity coverage per domain:")
print(coverage)
for dom in coverage.index:
    zero_cls = [c for c in coverage.columns if coverage.loc[dom, c] == 0]
    if zero_cls:
        print(f"  [note] domain '{dom}' has zero samples of {zero_cls} -- "
              f"mask these classes when reading that domain's per-class "
              f"accuracy, rather than treating 0% as failed transfer.")


In [ ]:
amp, phase = preprocess_measurement(read_measurement(index_df.iloc[0]["path"]), target_subcarriers=64)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].imshow(amp.T, aspect="auto", cmap="viridis"); axes[0].set_title("amplitude (preprocessed)")
axes[1].imshow(phase.T, aspect="auto", cmap="twilight"); axes[1].set_title("phase (preprocessed)")
for a in axes: a.set_xlabel("frame"); a.set_ylabel("subcarrier")
plt.tight_layout(); plt.show()


In [ ]:
"""PyTorch Dataset that ties the EHUNAM index + preprocessing + windowing
together. Builds a lightweight (file, start_frame) window index per domain
without holding all raw CSI in memory, and lazily reads/preprocesses on
__getitem__ (with an LRU cache so repeated windows from the same file
don't re-read/re-preprocess from disk each time)."""
from functools import lru_cache

import torch
import numpy as np
from torch.utils.data import Dataset


HEALTHCARE_ACTIVITIES = ["S", "T", "G", "F", "W"]  # standing, sitting, sit-to-stand, fall, walking


class EHUNAMWindowDataset(Dataset):
    def __init__(self, index_df, label_encoder, target_subcarriers=64,
                 win_len=64, stride=32, cache_size=8):
        self.df = index_df.reset_index(drop=True)
        self.label_encoder = label_encoder
        self.target_subcarriers = target_subcarriers
        self.win_len = win_len
        self.stride = stride

        # Build (row_idx, window_start) index using each file's frame count,
        # without loading CSI arrays yet. Uses the shared `window_starts`
        # helper (Section 3) instead of duplicating the arithmetic here --
        # the two copies had been able to drift apart (code review note).
        self.window_index = []
        for row_idx, row in self.df.iterrows():
            n = int(row["n_frames"])
            starts = window_starts(n, win_len, stride)
            self.window_index += [(row_idx, s) for s in starts]

        # Cache at least one slot per distinct file in this dataset. With
        # domains now correctly sized (dozens of files each, rather than
        # the pre-fix bug's ~1-file "domains"), holding every file's
        # preprocessed arrays in RAM after the first read is cheap and
        # avoids the constant cache-eviction/re-read-from-disk churn that
        # was contributing to slow epochs (code review, performance
        # section) -- especially the very first epoch, which pays for
        # every file's initial disk read no matter what.
        effective_cache_size = max(cache_size, self.df["path"].nunique())
        self._read_and_preprocess = lru_cache(maxsize=effective_cache_size)(self._read_and_preprocess_uncached)

        # Avalanche auto-infers per-sample labels from a `.targets` attribute
        # for plain (non-TensorDataset) torch Datasets -- without this it
        # can't build the task-aware classification dataset.
        self.targets = [
            self.label_encoder[self.df.loc[row_idx, "activity"]]
            for row_idx, _ in self.window_index
        ]

    def _read_and_preprocess_uncached(self, row_idx):
        path = self.df.loc[row_idx, "path"]
        rec = read_measurement(path)
        amp, phase = preprocess_measurement(rec, target_subcarriers=self.target_subcarriers)
        return amp, phase

    def __len__(self):
        return len(self.window_index)

    def __getitem__(self, i):
        row_idx, start = self.window_index[i]
        amp, phase = self._read_and_preprocess(row_idx)
        a = amp[start:start + self.win_len]
        p = phase[start:start + self.win_len]
        if a.shape[0] < self.win_len:
            pad = self.win_len - a.shape[0]
            a = np.pad(a, ((0, pad), (0, 0)), mode="edge")
            p = np.pad(p, ((0, pad), (0, 0)), mode="edge")
        x = torch.from_numpy(np.stack([a, p], axis=-1)).float()  # (win_len, S, 2)
        activity = self.df.loc[row_idx, "activity"]
        y = self.label_encoder[activity]
        return x, y


In [ ]:
TARGET_SUBCARRIERS = 64
WIN_LEN = 64
STRIDE = 32
TRAIN_FRAC = 0.8

label_list = sorted(index_df["activity"].unique())
label_encoder = {a: i for i, a in enumerate(label_list)}
N_CLASSES = len(label_list)
print("classes:", label_encoder)

# --- Class weighting (code review, section 4) ---
# Fall/Sit-to-stand/Sit are under-represented relative to Walk/Stand in
# MC1, and Fall is the class this healthcare framing cares about most --
# an unweighted CrossEntropyLoss has no reason to prioritize it. Compute
# inverse-frequency weights here; used by the criterion in Section 8 and by
# the plugin's replay loss in Section 7.
_class_counts = index_df["activity"].value_counts()
_counts_ordered = np.array([_class_counts.get(c, 0) for c in label_list], dtype=np.float64)
_counts_ordered = np.maximum(_counts_ordered, 1.0)  # guard divide-by-zero if a class is ever absent
_class_weights = _counts_ordered.sum() / (len(_counts_ordered) * _counts_ordered)
CLASS_WEIGHTS = torch.tensor(_class_weights, dtype=torch.float32)
print("\nPer-class weights (inverse-frequency):")
for c, w in zip(label_list, _class_weights):
    print(f"   {c}: weight={w:.3f}  (n={int(_class_counts.get(c, 0))})")


In [ ]:
# DIAGNOSTIC: Check what activities are actually in the (already
# HAR/activity-filtered) dataset. NOTE: this runs *after* the
# HEALTHCARE_ACTIVITIES filter in the cell above, not before it -- the
# original label here was misleading; kept as a running sanity check.
print("index_df shape (after Application==HAR + activity filtering):", index_df.shape)
print("\nUnique activities in dataset:")
activity_counts = index_df["activity"].value_counts().sort_values(ascending=False)
print(activity_counts)
print(f"\nTotal unique activities: {len(activity_counts)}")
print(f"\nHealthcare activities filter: {HEALTHCARE_ACTIVITIES}")
print(f"Rows that match healthcare filter: {len(index_df[index_df['activity'].isin(HEALTHCARE_ACTIVITIES)])}")


In [ ]:
# DEBUG: Test a handful of files end-to-end and confirm Environment now
# resolves to a real location, not a hash placeholder (code review check).
import glob
test_files = sorted(glob.glob(os.path.join(DATA_ROOT, "*.mat")))[:5]
print(f"Found {len(test_files)} files to test")

for test_file in test_files:
    print(f"\nTesting: {os.path.basename(test_file)}")
    try:
        rec = read_measurement(test_file)
        env_display = rec["environment"] if rec["environment"] else "(unresolved at file level -- filled in by build_index)"
        print(f"  \u2713 Loaded: activity={rec['activity']}, set={rec['set']}, "
              f"env={env_display}, shape={rec['csi'].shape}")
        if rec["environment"] and rec["environment"].startswith("env_"):
            print("  [!] This still looks like the old hash-based placeholder -- "
                  "the fix isn't taking effect for this file, check SET_TO_ENVIRONMENT "
                  "/ Summary.xlsx.")
    except Exception as e:
        print(f"  \u2717 Error: {type(e).__name__}: {str(e)[:100]}")


In [ ]:
try:
    from avalanche.benchmarks import benchmark_from_datasets
    from avalanche.benchmarks.utils import _make_taskaware_classification_dataset as make_taskaware_dataset
    AVALANCHE_AVAILABLE = True
except ImportError:
    AVALANCHE_AVAILABLE = False
    print("⚠ Avalanche not available - skipping benchmark creation.")
    print("  Install a C/C++ toolchain for your OS (see Section 1) and run: pip install avalanche-lib")
    print("  Proceeding with dataset preparation only.")

train_avl, test_avl, per_domain_datasets = [], [], []
for task_id, dom in enumerate(domain_order):
    dom_df = index_df[index_df["environment"] == dom].reset_index(drop=True)
    # Shuffle before the positional 80/20 split (code review, section 6) --
    # otherwise train/test is implicitly ordered by however glob happened
    # to sort filenames within this domain.
    dom_df = dom_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    n_train = max(1, int(len(dom_df) * TRAIN_FRAC))
    train_df, test_df = dom_df.iloc[:n_train], dom_df.iloc[n_train:]
    if len(test_df) == 0:
        print(f"  [warn] domain '{dom}' has only {len(dom_df)} file(s) -- too few "
              f"for an 80/20 split. Reusing the training file(s) as the test set "
              f"for this domain only; treat its per-domain accuracy as optimistic, "
              f"not a true generalization test (code review, section 6).")
        test_df = train_df

    train_ds = EHUNAMWindowDataset(train_df, label_encoder, target_subcarriers=TARGET_SUBCARRIERS,
                                    win_len=WIN_LEN, stride=STRIDE)
    test_ds = EHUNAMWindowDataset(test_df, label_encoder, target_subcarriers=TARGET_SUBCARRIERS,
                                   win_len=WIN_LEN, stride=STRIDE)
    per_domain_datasets.append((dom, train_ds, test_ds))
    print(f"task {task_id:>2} | {dom:<16} | train files={len(train_df):>4} test files={len(test_df):>4} "
          f"| train windows={len(train_ds):>5} | test windows={len(test_ds):>5}")

    if AVALANCHE_AVAILABLE:
        train_avl.append(make_taskaware_dataset(train_ds, task_labels=task_id))
        test_avl.append(make_taskaware_dataset(test_ds, task_labels=task_id))

if AVALANCHE_AVAILABLE:
    benchmark = benchmark_from_datasets(train=train_avl, test=test_avl)
    N_DOMAINS = len(domain_order)
    print(f"\nBuilt a {N_DOMAINS}-experience domain-incremental benchmark; experience 0 = base task ({domain_order[0]}).")
else:
    N_DOMAINS = len(domain_order)
    print(f"\n✓ Prepared {N_DOMAINS} domain datasets for training.")
    print(f"  Base task (experience 0): {domain_order[0]}")


In [ ]:
"""
CASCADE model building blocks.
Tested standalone before being transcribed into the notebook.
"""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------------------------
# 1. Selective State-Space (Mamba-lite) block
# ---------------------------------------------------------------------------
class SelectiveSSMBlock(nn.Module):
    """
    A simplified selective state-space sequence block (Mamba-style recurrence),
    implemented as a plain sequential scan in PyTorch (no custom CUDA kernel).
    Appropriate here because CSI windows are short (T ~ 25-100 frames), so the
    O(T) python-level loop over time is cheap and numerically transparent -
    this is the same reference recurrence used to *validate* fused Mamba
    kernels, just without the fused kernel.

        h_t = A_bar_t * h_{t-1} + dt_t * B_t * x_t      (elementwise per channel)
        y_t = (C_t * h_t).sum(state_dim) + D * x_t

    Shapes:
        input  x: (B, T, d_model)
        output y: (B, T, d_model)
    """

    def __init__(self, d_model: int, d_state: int = 16):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state

        # input-dependent (selective) parameters
        self.dt_proj = nn.Linear(d_model, d_model)
        self.B_proj = nn.Linear(d_model, d_model * d_state)
        self.C_proj = nn.Linear(d_model, d_model * d_state)

        # learnable per-channel state matrix diagonal (kept negative -> stable)
        self.A_log = nn.Parameter(torch.log(torch.rand(d_model, d_state) * 0.9 + 0.1))
        self.D = nn.Parameter(torch.ones(d_model))

        self.in_proj = nn.Linear(d_model, d_model)
        self.out_norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, Dm = x.shape
        assert Dm == self.d_model, f"expected last dim {self.d_model}, got {Dm}"
        Ds = self.d_state

        x_in = self.in_proj(x)  # (B,T,Dm)

        dt = F.softplus(self.dt_proj(x_in))                       # (B,T,Dm)
        Bt = self.B_proj(x_in).view(B, T, Dm, Ds)                 # (B,T,Dm,Ds)
        Ct = self.C_proj(x_in).view(B, T, Dm, Ds)                 # (B,T,Dm,Ds)

        A = -torch.exp(self.A_log)                                # (Dm,Ds) < 0
        # discretized transition, per step: (B,T,Dm,Ds)
        A_bar = torch.exp(dt.unsqueeze(-1) * A.unsqueeze(0).unsqueeze(0))

        h = x_in.new_zeros(B, Dm, Ds)
        ys = []
        for t in range(T):
            inp_t = dt[:, t].unsqueeze(-1) * Bt[:, t] * x_in[:, t].unsqueeze(-1)  # (B,Dm,Ds)
            h = A_bar[:, t] * h + inp_t
            y_t = (Ct[:, t] * h).sum(-1) + self.D * x_in[:, t]     # (B,Dm)
            ys.append(y_t)
        y = torch.stack(ys, dim=1)                                # (B,T,Dm)
        return self.out_norm(y + x)  # residual


# ---------------------------------------------------------------------------
# 2. Spatial "graph attention" over subcarriers/antennas
#    (multi-head attention + learned relative-distance bias -> lightweight
#     graph-attention without a torch_geometric dependency)
# ---------------------------------------------------------------------------
class GraphAttentionBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int = 4, max_rel_dist: int = 64):
        super().__init__()
        self.n_heads = n_heads
        self.d_model = d_model
        self.max_rel_dist = max_rel_dist
        self.rel_bias = nn.Embedding(2 * max_rel_dist + 1, n_heads)
        self.mha = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm = nn.LayerNorm(d_model)

    def _rel_bias_matrix(self, S: int, device):
        idx = torch.arange(S, device=device)
        rel = idx[None, :] - idx[:, None]
        rel = rel.clamp(-self.max_rel_dist, self.max_rel_dist) + self.max_rel_dist
        bias = self.rel_bias(rel)                      # (S,S,n_heads)
        return bias.permute(2, 0, 1)                    # (n_heads,S,S)

    def forward(self, node_feats: torch.Tensor) -> torch.Tensor:
        # node_feats: (B, S, d_model) -- S = number of subcarrier/antenna nodes
        B, S, D = node_feats.shape
        bias = self._rel_bias_matrix(S, node_feats.device)          # (n_heads,S,S)
        bias = bias.unsqueeze(0).expand(B, -1, -1, -1).reshape(B * self.n_heads, S, S)
        out, _ = self.mha(node_feats, node_feats, node_feats, attn_mask=None)
        # NOTE: nn.MultiheadAttention doesn't take a per-head additive bias
        # directly in old torch versions in a clean way, so we add the bias
        # effect via a lightweight secondary attention pass instead:
        out2 = out + self._bias_mix(node_feats, bias)
        return self.norm(out2 + node_feats)

    def _bias_mix(self, node_feats, bias):
        # bias: (B*n_heads, S, S) -- turn into an attention-weighted mix as a
        # cheap graph-structure prior added on top of content-based attention.
        B, S, D = node_feats.shape
        w = F.softmax(bias, dim=-1)                                  # (B*h,S,S)
        h = self.n_heads
        v = node_feats.view(B, S, h, D // h).permute(0, 2, 1, 3).reshape(B * h, S, D // h)
        mixed = torch.bmm(w, v)                                      # (B*h,S,D/h)
        mixed = mixed.view(B, h, S, D // h).permute(0, 2, 1, 3).reshape(B, S, D)
        return mixed


# ---------------------------------------------------------------------------
# 3. Relevance selection (gating) modules
# ---------------------------------------------------------------------------
class TemporalRelevanceSelection(nn.Module):
    """Learned gate approximating a mutual-information-style ranking: scores
    each time step, softmax over T, and produces both a pooled summary and
    re-weighted sequence."""

    def __init__(self, d_model: int):
        super().__init__()
        self.score = nn.Linear(d_model, 1)

    def forward(self, seq: torch.Tensor):
        # seq: (B,T,D)
        w = F.softmax(self.score(seq).squeeze(-1), dim=-1)   # (B,T)
        pooled = torch.einsum("bt,btd->bd", w, seq)
        reweighted = seq * w.unsqueeze(-1)
        return reweighted, pooled, w


class SpatialRelevanceSelection(nn.Module):
    """Attention-based gating over subcarrier/antenna nodes."""

    def __init__(self, d_model: int):
        super().__init__()
        self.score = nn.Linear(d_model, 1)

    def forward(self, nodes: torch.Tensor):
        # nodes: (B,S,D)
        w = F.softmax(self.score(nodes).squeeze(-1), dim=-1)  # (B,S)
        pooled = torch.einsum("bs,bsd->bd", w, nodes)
        reweighted = nodes * w.unsqueeze(-1)
        return reweighted, pooled, w


# ---------------------------------------------------------------------------
# 4. Cross-domain (temporal <-> spatial) fusion
# ---------------------------------------------------------------------------
class CrossDomainFusion(nn.Module):
    def __init__(self, d_model: int, n_heads: int = 4):
        super().__init__()
        self.t2s = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.s2t = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm_t = nn.LayerNorm(d_model)
        self.norm_s = nn.LayerNorm(d_model)

    def forward(self, temporal_seq: torch.Tensor, spatial_nodes: torch.Tensor):
        # temporal_seq: (B,T,D)  spatial_nodes: (B,S,D)
        t_att, _ = self.t2s(temporal_seq, spatial_nodes, spatial_nodes)
        s_att, _ = self.s2t(spatial_nodes, temporal_seq, temporal_seq)
        fused_t = self.norm_t(temporal_seq + t_att)
        fused_s = self.norm_s(spatial_nodes + s_att)
        return fused_t, fused_s  # (B,T,D), (B,S,D)


# ---------------------------------------------------------------------------
# 5. LoRA-style parameter-efficient adapter (for continual learner)
# ---------------------------------------------------------------------------
class LoRALinear(nn.Module):
    """Wraps a frozen nn.Linear with a trainable low-rank residual: adds
    (x @ A^T @ B^T) * scale to the frozen layer's output. Used to adapt the
    CASCADE backbone parameter-efficiently to new continual-learning tasks
    without touching (or storing new copies of) the base weights."""

    def __init__(self, base_linear: nn.Linear, r: int = 8, alpha: float = 16.0):
        super().__init__()
        self.base = base_linear
        for p in self.base.parameters():
            p.requires_grad_(False)
        in_f, out_f = base_linear.in_features, base_linear.out_features
        adapter_device = base_linear.weight.device
        adapter_dtype = base_linear.weight.dtype
        self.lora_A = nn.Parameter(torch.randn(r, in_f, device=adapter_device, dtype=adapter_dtype) * (1.0 / math.sqrt(r)))
        self.lora_B = nn.Parameter(torch.zeros(out_f, r, device=adapter_device, dtype=adapter_dtype))
        self.scale = alpha / r

    def forward(self, x):
        base_out = self.base(x)
        lora_out = F.linear(F.linear(x, self.lora_A), self.lora_B) * self.scale
        return base_out + lora_out


def inject_lora_adapters(module: nn.Module, r: int = 8, alpha: float = 16.0):
    """Recursively replace nn.Linear leaves with LoRALinear wrappers (freezing
    the originals), returning the list of newly-created adapter parameters.

    nn.MultiheadAttention sub-modules are deliberately left untouched: its
    fused forward path reads out_proj.weight / in_proj_weight directly rather
    than calling out_proj(x), so swapping in a generic Linear-wrapping
    adapter there would break it. Those layers stay frozen during the
    continual phase; adapters are applied to the SSM/gating/projection
    Linear layers instead.
    """
    new_params = []
    for name, child in list(module.named_children()):
        if isinstance(child, nn.MultiheadAttention):
            continue
        if isinstance(child, nn.Linear):
            wrapped = LoRALinear(child, r=r, alpha=alpha)
            setattr(module, name, wrapped)
            new_params += [wrapped.lora_A, wrapped.lora_B]
        else:
            new_params += inject_lora_adapters(child, r=r, alpha=alpha)
    return new_params


# ---------------------------------------------------------------------------
# 6. Gradient reversal layer + domain classifier (bias mitigation)
# ---------------------------------------------------------------------------
class _GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None


def grad_reverse(x, lambd=1.0):
    return _GradReverse.apply(x, lambd)


class BiasMitigationHead(nn.Module):
    """Domain-adversarial branch: tries to predict the domain/environment id
    from the shared representation through a gradient-reversal layer, so that
    (via minimax training) the backbone is pushed to make its features
    domain-invariant. Also exposes an attention-entropy regularizer that
    discourages the temporal/spatial gates from collapsing onto a narrow,
    potentially biased, subset of time steps / subcarriers."""

    def __init__(self, d_model: int, n_domains: int):
        super().__init__()
        self.domain_clf = nn.Sequential(
            nn.Linear(d_model, d_model // 2), nn.ReLU(), nn.Linear(d_model // 2, n_domains)
        )

    def forward(self, feats: torch.Tensor, lambd: float = 1.0):
        rev = grad_reverse(feats, lambd)
        domain_logits = self.domain_clf(rev)
        return domain_logits

    @staticmethod
    def attention_entropy_bonus(attn_weights: torch.Tensor, eps: float = 1e-8):
        # attn_weights: (B, L) already summing to 1 over L -- encourage higher
        # entropy (less peaky / less biased toward a few positions).
        ent = -(attn_weights * (attn_weights + eps).log()).sum(-1)
        return ent.mean()


In [ ]:
"""
Full CASCADE model assembled from the tested building blocks, plus the
latent-replay episodic memory and the Avalanche integration layer
(custom SupervisedTemplate subclass + custom continual-learning plugin).
"""
import random
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F



class BackboneBlock(nn.Module):
    """One hybrid block of the 'novel backbone': a selective-SSM sub-layer
    followed by a standard multi-head self-attention sub-layer (no CNN, no
    LSTM/BiLSTM anywhere in the stack)."""

    def __init__(self, d_model, n_heads=4, d_state=16):
        super().__init__()
        self.ssm = SelectiveSSMBlock(d_model, d_state=d_state)
        self.mha = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        x = self.ssm(x)
        att, _ = self.mha(x, x, x)
        return self.norm(x + att)


class CASCADE(nn.Module):
    """
    forward(x) where x: (B, T, S, 2)  [2 = amplitude, phase channels]
    returns a dict with logits / domain_logits / latent / attention diagnostics.

    Also exposes backbone_forward(x) -> latent and head_forward(latent) -> logits
    so that the continual-learning plugin can do *latent* replay (store/replay
    the small post-backbone vector rather than the raw CSI window).
    """

    def __init__(self, n_subcarriers, n_classes, n_domains, d_model=64,
                 n_temporal_layers=2, n_spatial_layers=2, n_backbone_layers=2,
                 d_state=16, n_heads=4):
        super().__init__()
        self.n_subcarriers = n_subcarriers
        self.d_model = d_model

        self.temporal_in = nn.Linear(n_subcarriers * 2, d_model)
        self.spatial_in = nn.Linear(4, d_model)  # mean/std of amp & phase per subcarrier

        self.temporal_layers = nn.ModuleList(
            [SelectiveSSMBlock(d_model, d_state=d_state) for _ in range(n_temporal_layers)]
        )
        self.spatial_layers = nn.ModuleList(
            [GraphAttentionBlock(d_model, n_heads=n_heads) for _ in range(n_spatial_layers)]
        )

        self.temporal_rel_sel = TemporalRelevanceSelection(d_model)
        self.spatial_rel_sel = SpatialRelevanceSelection(d_model)

        self.fusion = CrossDomainFusion(d_model, n_heads=n_heads)

        self.backbone = nn.ModuleList(
            [BackboneBlock(d_model, n_heads=n_heads, d_state=d_state) for _ in range(n_backbone_layers)]
        )
        self.pool_score = nn.Linear(d_model, 1)

        self.bias_head = BiasMitigationHead(d_model, n_domains=n_domains)
        self.recognition_head = nn.Linear(d_model, n_classes)

        # populated on every forward() call; read by the continual-learning
        # plugin (kept off the autograd-tracked `mb_output` that Avalanche
        # expects to be a plain logits tensor).
        self.last_aux = {}

    def backbone_forward(self, x: torch.Tensor):
        """x: (B,T,S,2) -> latent: (B,d_model). Everything up to (and
        including) backbone pooling -- this is the continual-learner 'split
        point': frozen after the base task except for the injected LoRA
        adapters."""
        B, T, S, _ = x.shape
        amp, phase = x[..., 0], x[..., 1]

        temporal_in = torch.cat([amp, phase], dim=-1)          # (B,T,2S)
        t = self.temporal_in(temporal_in)                       # (B,T,D)
        for layer in self.temporal_layers:
            t = layer(t)

        node_feats = torch.stack(
            [amp.mean(1), amp.std(1), phase.mean(1), phase.std(1)], dim=-1
        )                                                        # (B,S,4)
        s = self.spatial_in(node_feats)                          # (B,S,D)
        for layer in self.spatial_layers:
            s = layer(s)

        t_sel, pooled_t, w_t = self.temporal_rel_sel(t)
        s_sel, pooled_s, w_s = self.spatial_rel_sel(s)

        fused_t, fused_s = self.fusion(t_sel, s_sel)

        seq = torch.cat([fused_t, fused_s], dim=1)                # (B,T+S,D)
        for block in self.backbone:
            seq = block(seq)

        pool_w = F.softmax(self.pool_score(seq).squeeze(-1), dim=-1)  # (B,T+S)
        latent = torch.einsum("bl,bld->bd", pool_w, seq)

        self.last_aux = {"w_t": w_t.detach(), "w_s": w_s.detach()}
        self._last_latent_pre_head = latent
        return latent

    def head_forward(self, latent: torch.Tensor, lambd: float = 1.0):
        logits = self.recognition_head(latent)
        domain_logits = self.bias_head(latent, lambd=lambd)
        return logits, domain_logits

    def forward(self, x: torch.Tensor, lambd: float = 1.0):
        latent = self.backbone_forward(x)
        logits, domain_logits = self.head_forward(latent, lambd=lambd)
        self.last_aux.update({"latent": latent.detach(), "domain_logits": domain_logits})
        return logits


# ---------------------------------------------------------------------------
# Latent episodic / prototype memory
# ---------------------------------------------------------------------------
class LatentEpisodicMemory:
    """Reservoir-style buffer of *latent* vectors (not raw CSI), used for
    cheap latent replay, plus running per-class prototypes for unseen-
    scenario (OOD) scoring."""

    def __init__(self, max_per_class=200):
        self.max_per_class = max_per_class
        self.buffers = defaultdict(list)   # class_label -> list[(latent, domain_id)]
        self.seen_counts = defaultdict(int)

    def add(self, latents: torch.Tensor, labels: torch.Tensor, domain_id: int):
        for z, y in zip(latents, labels):
            y = int(y)
            self.seen_counts[y] += 1
            buf = self.buffers[y]
            if len(buf) < self.max_per_class:
                buf.append((z.clone(), domain_id))
            else:
                j = random.randint(0, self.seen_counts[y] - 1)
                if j < self.max_per_class:
                    buf[j] = (z.clone(), domain_id)

    def __len__(self):
        return sum(len(v) for v in self.buffers.values())

    def sample(self, n, return_domains=False):
        all_items = [(y, z, d) for y, buf in self.buffers.items() for (z, d) in buf]
        if not all_items:
            return None
        n = min(n, len(all_items))
        chosen = random.sample(all_items, n)
        ys = torch.tensor([c[0] for c in chosen])
        zs = torch.stack([c[1] for c in chosen])
        if return_domains:
            # code review, section 5: exposing each replayed latent's own
            # (past) domain id lets the plugin mix >1 domain into the
            # domain-adversarial loss instead of only ever seeing the
            # current experience's single domain per batch.
            ds = torch.tensor([c[2] for c in chosen])
            return zs, ys, ds
        return zs, ys

    def prototypes(self):
        protos = {}
        for y, buf in self.buffers.items():
            if buf:
                protos[y] = torch.stack([z for z, _ in buf]).mean(0)
        return protos

    def unseen_scenario_score(self, latent: torch.Tensor):
        """Mean distance from `latent` to the nearest class prototype --
        large values flag a likely unseen/OOD sensing scenario."""
        protos = self.prototypes()
        if not protos:
            return torch.zeros(latent.shape[0])
        P = torch.stack(list(protos.values()))                # (C,D)
        d = torch.cdist(latent, P)                             # (B,C)
        return d.min(dim=-1).values


In [ ]:
# quick sanity check: one forward + backward pass with dummy data shaped
# like a real batch from the benchmark above, including the Phase 2 LoRA transition.
_dbg_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_dbg_model = CASCADE(n_subcarriers=64, n_classes=5, n_domains=5, d_model=64).to(_dbg_device)
for _dbg_param in _dbg_model.parameters():
    _dbg_param.requires_grad_(False)
inject_lora_adapters(_dbg_model.temporal_layers)
_dbg_lora = next(
    _dbg_param for _dbg_param in _dbg_model.parameters()
    if _dbg_param.requires_grad
)
assert _dbg_lora.device == next(_dbg_model.parameters()).device
_x = torch.randn(4, 64, 64, 2, device=_dbg_device)
_logits = _dbg_model(_x)
_loss = F.cross_entropy(_logits, torch.randint(0, 5, (4,), device=_dbg_device))
_loss.backward()
print("sanity check OK -- LoRA device:", _dbg_lora.device, "logits:", _logits.shape, "loss:", float(_loss.detach()))
del _dbg_model, _dbg_lora, _dbg_param, _x, _logits, _loss


In [ ]:
"""
Avalanche integration for CASCADE:
  - CascadeTemplate: SupervisedTemplate subclass that knows how to call a
    model whose forward() takes (x, lambd) and stashes auxiliary outputs on
    the model rather than in mb_output (mb_output must stay a plain logits
    tensor for Avalanche's built-in accuracy/loss/forgetting metrics to work).
  - CascadeContinualPlugin: implements the "Continual Learner" box from the
    figure -- after the base task (experience 0) it freezes the backbone and
    injects LoRA adapters, and on every subsequent experience it (a) adds a
    domain-adversarial loss + attention-entropy bonus, and (b) mixes in a
    *latent* replay loss sampled from the episodic memory.
"""
import torch
import torch.nn.functional as F

from avalanche.training.templates import SupervisedTemplate
from avalanche.core import SupervisedPlugin



class CascadeTemplate(SupervisedTemplate):
    """Only the forward() pass is overridden; criterion() stays the default
    (CE on mb_output vs mb_y) so all of Avalanche's built-in metrics keep
    working unmodified. Auxiliary losses are added by CascadeContinualPlugin
    via before_backward."""

    def forward(self):
        logits = self.model(self.mb_x)
        return logits


class CascadeContinualPlugin(SupervisedPlugin):
    def __init__(self, memory: LatentEpisodicMemory, lora_r=8, lora_alpha=16.0,
                 replay_weight=1.0, replay_batch_size=32,
                 domain_weight=0.1, entropy_weight=0.01,
                 freeze_backbone_after_first_exp=True,
                 latent_dump_batches=4, class_weights=None):
        super().__init__()
        self.memory = memory
        self.lora_r = lora_r
        self.lora_alpha = lora_alpha
        self.replay_weight = replay_weight
        self.replay_batch_size = replay_batch_size
        self.domain_weight = domain_weight
        self.entropy_weight = entropy_weight
        self.freeze_backbone_after_first_exp = freeze_backbone_after_first_exp
        self.latent_dump_batches = latent_dump_batches
        # code review, section 4: same per-class weights used by the main
        # criterion, applied here too since the replay loss is computed
        # manually (not routed through strategy.criterion).
        self.class_weights = class_weights
        self._adapters_injected = False

    # ---- Phase transition: base task -> continual (parameter-efficient) ----
    def before_training_exp(self, strategy, **kwargs):
        exp_id = strategy.experience.current_experience
        if (exp_id >= 1 and self.freeze_backbone_after_first_exp
                and not self._adapters_injected):
            model = strategy.model
            for p in model.parameters():
                p.requires_grad_(False)
            new_params = []
            for submodule in [model.temporal_in, model.spatial_in,
                               model.temporal_layers, model.spatial_layers,
                               model.fusion, model.backbone]:
                new_params += inject_lora_adapters(submodule, r=self.lora_r, alpha=self.lora_alpha)
            # keep heads + bias-mitigation branch trainable (cheap, task-specific).
            # NOTE: these params were already registered in the optimizer at
            # construction time (when requires_grad was still True for
            # everything) -- flipping requires_grad back on is all that's
            # needed for them to receive gradients again; re-adding them as a
            # *new* param group would duplicate them and raise a ValueError.
            for p in model.recognition_head.parameters():
                p.requires_grad_(True)
            for p in model.bias_head.parameters():
                p.requires_grad_(True)
            strategy.optimizer.add_param_group({"params": new_params})
            self._adapters_injected = True
            print(f"[CascadeContinualPlugin] experience {exp_id}: backbone frozen, "
                  f"{sum(p.numel() for p in new_params)} LoRA params added.")

    # ---- Auxiliary losses: domain-adversarial + entropy + latent replay ----
    def before_backward(self, strategy, **kwargs):
        model = strategy.model
        aux = model.last_aux

        ent = BiasMitigationHead.attention_entropy_bonus(aux["w_t"]) \
            + BiasMitigationHead.attention_entropy_bonus(aux["w_s"])
        strategy.loss = strategy.loss - self.entropy_weight * ent

        # Domain-adversarial term. NOTE (code review, section 5): every
        # mini-batch drawn from the *current* experience only ever contains
        # one domain id, so on its own this term gives the domain
        # classifier nothing to discriminate against -- it trivially learns
        # "always predict the one label in this batch" and gradient
        # reversal against an already-near-zero loss does little. Mixing
        # the replayed samples' own (past) domain ids into the same
        # cross-entropy call gives it a real multi-domain signal from
        # experience 1 onward, when memory has content; experience 0 is
        # unavoidably single-domain (there's nothing to replay yet).
        domain_logits_list = [aux["domain_logits"]]
        domain_labels_list = [strategy.mb_task_id.to(strategy.device)]

        if strategy.experience.current_experience >= 1 and len(self.memory) > 0:
            sampled = self.memory.sample(self.replay_batch_size, return_domains=True)
            if sampled is not None:
                zs, ys, ds = sampled
                zs = zs.to(strategy.device)
                ys = ys.to(strategy.device)
                ds = ds.to(strategy.device)
                logits_r, domain_logits_r = model.head_forward(zs)
                replay_loss = F.cross_entropy(logits_r, ys, weight=self.class_weights)
                strategy.loss = strategy.loss + self.replay_weight * replay_loss
                domain_logits_list.append(domain_logits_r)
                domain_labels_list.append(ds)

        domain_logits_all = torch.cat(domain_logits_list, dim=0)
        domain_labels_all = torch.cat(domain_labels_list, dim=0)
        domain_loss = F.cross_entropy(domain_logits_all, domain_labels_all)
        strategy.loss = strategy.loss + self.domain_weight * domain_loss

    # ---- populate latent memory with the task we just finished ----
    def after_training_exp(self, strategy, **kwargs):
        model = strategy.model
        model.eval()
        exp_id = strategy.experience.current_experience
        loader = torch.utils.data.DataLoader(
            strategy.experience.dataset, batch_size=strategy.train_mb_size, shuffle=True
        )
        with torch.no_grad():
            for i, batch in enumerate(loader):
                if i >= self.latent_dump_batches:
                    break
                x, y = batch[0].to(strategy.device), batch[1]
                latent = model.backbone_forward(x)
                self.memory.add(latent.cpu(), y, domain_id=exp_id)
        model.train()
        print(f"[CascadeContinualPlugin] experience {exp_id} done -> memory size {len(self.memory)}")


In [ ]:
D_MODEL = 128
N_TEMPORAL_LAYERS = 2
N_SPATIAL_LAYERS = 2
N_BACKBONE_LAYERS = 3
D_STATE = 16
N_HEADS = 4

LORA_R = 8
LORA_ALPHA = 16.0
REPLAY_WEIGHT = 1.0
REPLAY_BATCH_SIZE = 32
DOMAIN_ADV_WEIGHT = 0.1
ENTROPY_WEIGHT = 0.01
MEMORY_PER_CLASS = 200

TRAIN_MB_SIZE = 32
TRAIN_EPOCHS_PHASE1 = 100     # base-task offline training -- give this one more epochs
TRAIN_EPOCHS_PHASE2 = 100     # continual steps -- lighter, adapters do the heavy lifting
LR = 1e-3


In [ ]:
from avalanche.training.plugins import EvaluationPlugin
from avalanche.evaluation.metrics import accuracy_metrics, forgetting_metrics, loss_metrics
from avalanche.logging import InteractiveLogger

model = CASCADE(n_subcarriers=TARGET_SUBCARRIERS, n_classes=N_CLASSES, n_domains=N_DOMAINS,
                 d_model=D_MODEL, n_temporal_layers=N_TEMPORAL_LAYERS, n_spatial_layers=N_SPATIAL_LAYERS,
                 n_backbone_layers=N_BACKBONE_LAYERS, d_state=D_STATE, n_heads=N_HEADS).to(DEVICE)

optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LR)

# code review, section 4: apply the per-class weights computed in Section 5
# to both the main criterion and the plugin's replay loss.
class_weights = CLASS_WEIGHTS.to(DEVICE)

memory = LatentEpisodicMemory(max_per_class=MEMORY_PER_CLASS)
cl_plugin = CascadeContinualPlugin(
    memory=memory, lora_r=LORA_R, lora_alpha=LORA_ALPHA,
    replay_weight=REPLAY_WEIGHT, replay_batch_size=REPLAY_BATCH_SIZE,
    domain_weight=DOMAIN_ADV_WEIGHT, entropy_weight=ENTROPY_WEIGHT,
    class_weights=class_weights,
)

eval_plugin = EvaluationPlugin(
    accuracy_metrics(experience=True, stream=True),
    loss_metrics(stream=True),
    forgetting_metrics(experience=True, stream=True),
    loggers=[InteractiveLogger()],
)

strategy = CascadeTemplate(
    model=model, optimizer=optimizer, criterion=nn.CrossEntropyLoss(weight=class_weights),
    train_mb_size=TRAIN_MB_SIZE, train_epochs=TRAIN_EPOCHS_PHASE1,
    eval_mb_size=TRAIN_MB_SIZE, device=DEVICE,
    plugins=[cl_plugin], evaluator=eval_plugin,
)


In [ ]:
results_per_experience = []

for exp in benchmark.train_stream:
    exp_id = exp.current_experience
    if exp_id == 1:
        # entering the continual phase: fewer epochs per task is typical
        # once adapters + replay are doing the work instead of a full
        # from-scratch fit.
        strategy.train_epochs = TRAIN_EPOCHS_PHASE2
    phase_name = "PHASE 1 (base task, full backbone)" if exp_id == 0 else "PHASE 2 (continual: LoRA + latent replay)"
    print(f"\n{'='*70}\nExperience {exp_id} -- domain '{domain_order[exp_id]}' -- {phase_name}\n{'='*70}")

    strategy.train(exp)
    res = strategy.eval(benchmark.test_stream)
    results_per_experience.append((exp_id, res))

print("\nTraining complete.")


In [ ]:
def extract_acc_matrix(results_per_experience, n_domains):
    mat = np.full((n_domains, n_domains), np.nan)
    for trained_up_to, res in results_per_experience:
        for j in range(n_domains):
            key = f"Top1_Acc_Exp/eval_phase/test_stream/Exp{j:03d}"
            if key in res:
                mat[trained_up_to, j] = res[key]
    return mat

acc_matrix = extract_acc_matrix(results_per_experience, N_DOMAINS)

fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(acc_matrix, vmin=0, vmax=1, cmap="viridis")
ax.set_xticks(range(N_DOMAINS)); ax.set_xticklabels(domain_order, rotation=45, ha="right")
ax.set_yticks(range(N_DOMAINS)); ax.set_yticklabels([f"after task {i}" for i in range(N_DOMAINS)])
ax.set_xlabel("evaluated on domain"); ax.set_title("Continual-learning accuracy matrix")
for i in range(N_DOMAINS):
    for j in range(N_DOMAINS):
        if not np.isnan(acc_matrix[i, j]):
            ax.text(j, i, f"{acc_matrix[i,j]:.2f}", ha="center", va="center",
                    color="white" if acc_matrix[i, j] < 0.6 else "black", fontsize=8)
plt.colorbar(im, ax=ax, label="Top-1 accuracy")
plt.tight_layout(); plt.show()


In [ ]:
stream_acc = [res.get("Top1_Acc_Stream/eval_phase/test_stream") for _, res in results_per_experience]
stream_forgetting = [res.get("StreamForgetting/eval_phase/test_stream") for _, res in results_per_experience]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].plot(range(N_DOMAINS), stream_acc, marker="o"); axes[0].set_title("Stream accuracy after each experience")
axes[0].set_xlabel("experience"); axes[0].set_ylabel("accuracy"); axes[0].set_ylim(0, 1)
axes[1].plot(range(N_DOMAINS), stream_forgetting, marker="o", color="tomato"); axes[1].set_title("Stream forgetting")
axes[1].set_xlabel("experience"); axes[1].set_ylabel("forgetting")
plt.tight_layout(); plt.show()


In [ ]:
# code review, section 8: this was hardcoded to a Colab Google-Drive path
# ("/content/drive/MyDrive/...") but the rest of this notebook (DATA_ROOT,
# etc.) shows it's actually being run locally -- that path doesn't exist
# outside Colab and this cell would fail. Use a local (or env-var
# overridable) path instead; if you genuinely are on Colab with Drive
# mounted, set EHUNAM_CKPT_DIR accordingly before running this cell.
CKPT_DIR = os.environ.get("EHUNAM_CKPT_DIR", "./checkpoints/EHUNAM")
os.makedirs(CKPT_DIR, exist_ok=True)

torch.save({
    "model_state_dict": model.state_dict(),
    "label_encoder": label_encoder,
    "domain_order": domain_order,
    "config": dict(d_model=D_MODEL, n_temporal_layers=N_TEMPORAL_LAYERS, n_spatial_layers=N_SPATIAL_LAYERS,
                    n_backbone_layers=N_BACKBONE_LAYERS, d_state=D_STATE, n_heads=N_HEADS,
                    n_subcarriers=TARGET_SUBCARRIERS, n_classes=N_CLASSES, n_domains=N_DOMAINS),
}, os.path.join(CKPT_DIR, "cascade_checkpoint.pt"))

print("saved to", os.path.join(CKPT_DIR, "cascade_checkpoint.pt"))
